In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:
df = pd.read_csv("juliet_cwe_dataset.csv")
print(len(df))

179839


In [3]:
df.memory_usage(deep=True).sum() / (1024**3)

np.float64(1.1109247040003538)

In [4]:
# ============================================
# 🧠 PHASE 3: MODEL TRAINING & EVALUATION
# Multi-class CWE Classification (118 classes)
# Model: CodeT5-small
# ============================================

# import os
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, confusion_matrix
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
# from tqdm import tqdm

# -----------------------------
# 1. CONFIG
# -----------------------------
CSV_PATH = "/Users/rohanbabbar/Documents/Final Year/cwe/Archive/juliet_cwe_dataset.csv"
MODEL_NAME = "Salesforce/codet5-small"
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 5e-5

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"🚀 Using device: {device}")

# -----------------------------
# 2. LOAD DATA
# -----------------------------
df = pd.read_csv(CSV_PATH)

# Keep only needed columns
df = df[["code_clean", "cwe_index"]].dropna()
df["cwe_index"] = df["cwe_index"].astype(int)

print("📦 Dataset loaded")
print(f"Total samples: {len(df)}")
print(df["cwe_index"].value_counts().head())

# Train/validation split (stratified by CWE)
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["cwe_index"]
)

print(f"\n🧪 Train samples: {len(train_df)}")
print(f"🧪 Val samples:   {len(val_df)}")

# -----------------------------
# 3. TOKENIZER
# -----------------------------
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)
print("\n✅ Tokenizer loaded")

# -----------------------------
# 4. CUSTOM DATASET
# -----------------------------
class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        code = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            code,
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )

        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label)
        return item

train_dataset = JulietDataset(
    train_df["code_clean"],
    train_df["cwe_index"],
    tokenizer,
    max_len=MAX_LENGTH
)

val_dataset = JulietDataset(
    val_df["code_clean"],
    val_df["cwe_index"],
    tokenizer,
    max_len=MAX_LENGTH
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("\n✅ Datasets & dataloaders ready")

# -----------------------------
# 5. MODEL & OPTIMIZER
# -----------------------------
num_labels = df["cwe_index"].nunique()
print(f"\n🔢 Number of CWE classes: {num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=LR)
print("✅ Model & optimizer ready")

# -----------------------------
# 6. TRAINING LOOP
# -----------------------------
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def eval_model(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, all_labels, all_preds


for epoch in range(EPOCHS):
    print(f"\n================= EPOCH {epoch+1}/{EPOCHS} =================")

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    print(f"📉 Train loss: {train_loss:.4f}")

    val_loss, y_true, y_pred = eval_model(model, val_loader, device)
    print(f"📉 Val loss:   {val_loss:.4f}")

    print("\n📊 Classification Report (Validation):")
    print(classification_report(y_true, y_pred, digits=4))

    print("📊 Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

print("\n✅ Training complete!")

# -----------------------------
# 7. SAVE MODEL
# -----------------------------
SAVE_DIR = "codet5_juliet_multiclass"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\n💾 Model & tokenizer saved to: {SAVE_DIR}")

🚀 Using device: mps
📦 Dataset loaded
Total samples: 179839
cwe_index
2      16314
1      14938
110    14844
11     12282
7       9150
Name: count, dtype: int64

🧪 Train samples: 143871
🧪 Val samples:   35968


/Users/rohanbabbar/miniconda3/envs/cwe/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✅ Tokenizer loaded

✅ Datasets & dataloaders ready

🔢 Number of CWE classes: 118


/Users/rohanbabbar/miniconda3/envs/cwe/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

📉 Val loss:   0.0052

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       356
           1     1.0000    0.9993    0.9997      2988
           2     0.9988    0.9997    0.9992      3263
           3     1.0000    1.0000    1.0000        92
           4     1.0000    0.9975    0.9987      1191
           5     1.0000    1.0000    1.0000       831
           6     1.0000    1.0000    1.0000      1191
           7     1.0000    0.9973    0.9986      1830
           8     1.0000    1.0000    1.0000        30
           9     1.0000    1.0000    1.0000        31
          10     1.0000    1.0000    1.0000        16
          11     1.0000    0.9980    0.9990      2457
          12     1.0000    0.9983    0.9992      1816
          13     0.9519    1.0000    0.9754       713
          14     1.0000    0.9874    0.9936       713
          15     1.0000    1.0000    1.0000         8
          16     1.00

Training: 100%|██████████| 35968/35968 [2:38:11<00:00,  3.79it/s]  


📉 Train loss: 0.0052


Evaluating: 100%|██████████| 8992/8992 [11:04<00:00, 13.53it/s]


📉 Val loss:   0.0045

📊 Classification Report (Validation):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       356
           1     1.0000    0.9993    0.9997      2988
           2     1.0000    0.9991    0.9995      3263
           3     1.0000    1.0000    1.0000        92
           4     0.9975    1.0000    0.9987      1191
           5     1.0000    1.0000    1.0000       831
           6     1.0000    1.0000    1.0000      1191
           7     1.0000    1.0000    1.0000      1830
           8     1.0000    1.0000    1.0000        30
           9     1.0000    1.0000    1.0000        31
          10     1.0000    1.0000    1.0000        16
          11     1.0000    0.9980    0.9990      2457
          12     0.9956    0.9989    0.9973      1816
          13     1.0000    0.9902    0.9951       713
          14     1.0000    0.9874    0.9936       713
          15     1.0000    1.0000    1.0000         8
          16     0.95

In [7]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import torch
from tqdm import tqdm

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating on unseen data"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

# Run evaluation
y_true, y_pred = evaluate_model(model, val_loader, device)

print("\n📊 Classification Report (Unseen Validation Data):")
print(classification_report(y_true, y_pred, digits=4))

print("\n📉 Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Evaluating on unseen data: 100%|██████████| 8992/8992 [11:04<00:00, 13.53it/s]


📊 Classification Report (Unseen Validation Data):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       356
           1     0.9990    0.9997    0.9993      2988
           2     1.0000    0.9991    0.9995      3263
           3     1.0000    1.0000    1.0000        92
           4     1.0000    0.9975    0.9987      1191
           5     1.0000    1.0000    1.0000       831
           6     1.0000    1.0000    1.0000      1191
           7     0.9984    1.0000    0.9992      1830
           8     1.0000    1.0000    1.0000        30
           9     1.0000    1.0000    1.0000        31
          10     1.0000    1.0000    1.0000        16
          11     1.0000    0.9980    0.9990      2457
          12     1.0000    0.9983    0.9992      1816
          13     0.9958    1.0000    0.9979       713
          14     0.9674    1.0000    0.9834       713
          15     1.0000    1.0000    1.0000         8
          16     1.0000    0.9

In [8]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    y_true,
    y_pred,
    digits=4,
    output_dict=True
)
report_df = pd.DataFrame(report).transpose()
report_df.reset_index(inplace=True)
report_df.rename(columns={"index": "cwe_index"}, inplace=True)

# Keep only actual classes (remove accuracy / avg rows)
report_df = report_df[
    report_df["cwe_index"].str.isnumeric()
]
report_df["cwe_index"] = report_df["cwe_index"].astype(int)
# Load mapping
mapping_df = pd.read_csv("/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv")
cwe_map = mapping_df[["cwe_index", "cwe"]].drop_duplicates()

report_df = report_df.merge(cwe_map, on="cwe_index", how="left")
train_counts = train_df["cwe_index"].value_counts().to_dict()
val_counts = val_df["cwe_index"].value_counts().to_dict()

report_df["train_samples"] = report_df["cwe_index"].map(train_counts)
report_df["validation_samples"] = report_df["cwe_index"].map(val_counts)
final_df = report_df[[
    "cwe",
    "train_samples",
    "validation_samples",
    "precision",
    "recall",
    "f1-score",
    "support"
]]

final_df.rename(columns={"f1-score": "f1"}, inplace=True)
final_df.to_csv("cwe_metrics_summary.csv", index=False)
print("✅ Saved cwe_metrics_summary.csv")

✅ Saved cwe_metrics_summary.csv


/var/folders/3z/ksnlkp8d7d12pcm61cnddf3h0000gn/T/ipykernel_37593/416650136.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns={"f1-score": "f1"}, inplace=True)
